since the bronze data was read in multiple batches, it's good to optimize the delta table before using it as a spark table

In [0]:
spark.sql("OPTIMIZE ca_healthcare_fac_bronze.ca_general_land_use_data.ca_land_use_df")

In [0]:
import pandas as pd
# from arcgis.features import GeoAccessor, GeoSeriesAccessor

ca_land_bronze_path = "ca_healthcare_fac_bronze.ca_general_land_use_data.ca_land_use_df"

# read the entire bronze table table first then filter for open space and public land
ca_land_bronze_df = spark.read.table(ca_land_bronze_path)

ca_land_bronze_filtered = ca_land_bronze_df.filter(ca_land_bronze_df.ucd_description == "Open space and public lands")

ca_land_public_space_pddf = ca_land_bronze_filtered.toPandas()

ca_land_public_space_pddf.shape

In [0]:
from shapely.geometry import Polygon, Point

def get_centroid(rings):
    ring = rings[0]
    polygon = Polygon(ring)
    return Point(polygon.centroid)

ca_land_public_space_pddf[['centroid']] = ca_land_public_space_pddf['geometry_rings'].apply(lambda rings: pd.Series(get_centroid(rings=rings)))
ca_land_public_space_pddf.head()




Lookup the centroid latidue and longitude and get the mssa object.

First, convert the geometry rings to polygons

Second, shape join the 2 dataframes and bring in Tractce, geoid, mssaid, mssanm, definition columns

In [0]:
mssa_bronze_pddf = spark.read.table("ca_healthcare_fac_bronze.mssa_data_bronze.mssa_geo").toPandas()
print(mssa_bronze_pddf.columns)

In [0]:
from shapely.geometry import Polygon

def convert_polygon(rings):
    return Polygon(rings[0])

mssa_bronze_pddf['geometry_rings'] = mssa_bronze_pddf['geometry_rings'].apply(convert_polygon)
mssa_bronze_pddf.head()


In [0]:
mssa_bronze_pddf.dtypes

In [0]:
mssa_bronze_pddf.columns

In [0]:
import geopandas as gpd

ca_land_gdf = gpd.GeoDataFrame(ca_land_public_space_pddf, geometry='centroid', crs='EPSG:4326')
mssa_geo_gdf = gpd.GeoDataFrame(mssa_bronze_pddf[['TRACTCE', 'GEOID', 'MSSAID', 'MSSANM', 'geometry_rings']], geometry='geometry_rings', crs='EPSG:4326')

ca_land_space_with_mssa = ca_land_gdf.sjoin_nearest(mssa_geo_gdf, how='left')
ca_land_space_with_mssa.head()
ca_land_space_with_mssa.shape

In [0]:
ca_land_space_with_mssa = ca_land_space_with_mssa.drop_duplicates(subset=['OBJECTID', 'MSSAID'])
ca_land_space_with_mssa.shape
ca_land_space_with_mssa.head(100)




In [0]:
ca_land_space_with_mssa['centroid_lat'] = ca_land_space_with_mssa['centroid'].apply(
    lambda g: g.y if g is not None else None
)
ca_land_space_with_mssa['centroid_lon'] = ca_land_space_with_mssa['centroid'].apply(
    lambda g: g.x if g is not None else None
)
ca_land_space_with_mssa.drop(['centroid'], axis=1, inplace=True)
ca_land_space_with_mssa.head(100)


In [0]:
ca_land_silver_df = spark.createDataFrame(ca_land_space_with_mssa)
ca_land_silver_df.write.mode('overwrite').saveAsTable('ca_healthcare_fac_silver.ca_public_land_mssa_silver.ca_public_land_mssa')